# Study2 Close Price Forecasting and Framework Validation

This notebook validates Brain-AI time-series backend availability and runs forecasting experiments using the last 1 year as test horizon.

It includes:
- Framework backend status and TimeSeriesAutoML execution
- Statistical and decomposition-based forecasting models
- Comprehensive forecast error metrics
- Forecast vs actual visualization for one stock
- Auto-generation of modality-specific notebook templates

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make local source importable without requiring pip install -e .
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from brain_automl.config import get_default_config
from brain_automl.core import BACKEND_REGISTRY
from brain_automl.model_zoo.time_series_ai import TimeSeriesAutoML

warnings.filterwarnings("ignore")

DATA_PATH = Path.cwd() / "study2_volatility_dataset.csv"
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Data exists: {DATA_PATH.exists()}")

/Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-03 01:03:06,055	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-04-03 01:03:06,121	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Project root: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai
Data path: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/study2_volatility_dataset.csv
Data exists: True


In [2]:
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

print(f"Shape: {df.shape}")
print("Columns:", list(df.columns))

DATE_COL = "Date"
TARGET_COL = "Close"
GROUP_COL = "Stock" if "Stock" in df.columns else None

if GROUP_COL:
    stock_counts = df[GROUP_COL].value_counts().head(10)
    print("Top stocks by row count:")
    display(stock_counts.to_frame("rows"))

display(df.head())

Shape: (18381, 12)
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Return', 'MA_5', 'MA_10', 'MA_20', 'Volatility', 'Stock']
Top stocks by row count:


,rows
Stock,
HDFCBANK,3681
TCS,3681
RELIANCE,3681
BANKNIFTY,3677
NIFTY50,3661


,Date,Close,High,Low,Open,Volume,Return,MA_5,MA_10,MA_20,Volatility,Stock
0,2010-02-01,70.289337,71.530576,69.641253,71.530576,15034660,-0.019376,70.985294,73.677579,74.229110,0.000375,HDFCBANK
1,2010-02-02,69.382011,70.954982,69.070051,70.607870,20669400,-0.012992,70.303383,72.853526,73.950984,0.000169,HDFCBANK
2,2010-02-03,71.299896,71.723900,69.333688,70.300316,15455340,0.027267,70.527908,72.166122,73.765456,0.000744,HDFCBANK
3,2010-02-04,71.036263,71.394356,69.663210,71.091185,17210480,-0.003704,70.734415,71.553850,73.564221,0.000014,HDFCBANK
4,2010-02-05,69.129372,70.212434,68.762489,70.212434,18693880,-0.027211,70.227376,70.946192,73.257866,0.000740,HDFCBANK


In [3]:
# Use one stock for a univariate forecasting benchmark.
work_df = df.copy()
work_df[DATE_COL] = pd.to_datetime(work_df[DATE_COL], errors="coerce")
work_df = work_df.dropna(subset=[DATE_COL, TARGET_COL])

if GROUP_COL:
    selected_stock = work_df[GROUP_COL].iloc[0]
    work_df = work_df[work_df[GROUP_COL] == selected_stock].copy()
else:
    selected_stock = "single_series"

ts_df = work_df[[DATE_COL, TARGET_COL]].sort_values(DATE_COL).reset_index(drop=True)
ts_df = ts_df.rename(columns={DATE_COL: "ds", TARGET_COL: "y"})

TEST_HORIZON = min(252, max(30, len(ts_df) // 5))
train_df = ts_df.iloc[:-TEST_HORIZON].copy()
test_df = ts_df.iloc[-TEST_HORIZON:].copy()

print(f"Selected stock: {selected_stock}")
print(f"Total rows: {len(ts_df)}")
print(f"Train rows: {len(train_df)}")
print(f"Test rows (last ~1 year): {len(test_df)}")
print(f"Train range: {train_df['ds'].min().date()} -> {train_df['ds'].max().date()}")
print(f"Test range: {test_df['ds'].min().date()} -> {test_df['ds'].max().date()}")

Selected stock: HDFCBANK
Total rows: 3681
Train rows: 3429
Test rows (last ~1 year): 252
Train range: 2010-02-01 -> 2023-12-20
Test range: 2023-12-21 -> 2024-12-31


In [4]:
# Check configured framework backends and direct import availability.
config = get_default_config()
configured_backends = config["backends"]["by_modality"]["time_series"]["default"]

direct_import_checks = {
    "autogluon_timeseries": "autogluon.timeseries",
    "statsforecast": "statsforecast",
    "neuralforecast": "neuralforecast",
    "flaml": "flaml",
}

availability_rows = []
for b in configured_backends:
    registered = BACKEND_REGISTRY.has(b)
    backend_available = False
    if registered:
        backend_cls = BACKEND_REGISTRY.get(b)
        backend_available = bool(backend_cls.is_available())

    import_name = direct_import_checks.get(b, b)
    direct_import_available = False
    try:
        __import__(import_name)
        direct_import_available = True
    except Exception:
        direct_import_available = False

    availability_rows.append({
        "backend": b,
        "registered": registered,
        "framework_import_available": backend_available,
        "direct_import_available": direct_import_available,
    })

availability_df = pd.DataFrame(availability_rows)
display(availability_df)

available_backends = availability_df.loc[availability_df["registered"] & availability_df["framework_import_available"], "backend"].tolist()
print("Available backends for framework execution:", available_backends)

,backend,registered,framework_import_available,direct_import_available
0,autogluon_timeseries,True,True,True
1,statsforecast,True,True,True
2,neuralforecast,True,True,True
3,flaml,True,True,True


Available backends for framework execution: ['autogluon_timeseries', 'statsforecast', 'neuralforecast', 'flaml']


In [ ]:
# Run all available backends via the high-level forecast_last_horizon API.
# This handles standardisation, train/test split, per-backend execution, and
# metric computation internally — no manual reshaping needed.
run_config = get_default_config()
run_config["backends"]["hard_fail_if_no_backend_available_for_modality"] = False
run_config["modalities"]["allow_partial_success"] = True

executor = TimeSeriesAutoML(config=run_config)
framework_output = executor.forecast_last_horizon(
    data=df,
    timestamp_column=DATE_COL,
    target_column=TARGET_COL,
    item_id_column=GROUP_COL,
    horizon=TEST_HORIZON,
    backends=available_backends,
)

print(f"Item       : {framework_output['item_id']}")
print(f"Frequency  : {framework_output['frequency']}")
print(f"Horizon    : {framework_output['horizon']}")
print(f"Backends   : {[r.backend for r in framework_output['results']]}")
statuses = {r.backend: r.metadata.get('status', 'unknown') for r in framework_output['results']}
print(f"Statuses   : {statuses}")


Seed set to 1


In [ ]:
# Display Brain-AI framework backend metrics table (sorted by RMSE).
fw_metrics = framework_output["metrics"]

if fw_metrics is not None and not fw_metrics.empty:
    display_cols = [c for c in ["backend", "mae", "rmse", "mape", "smape", "mase", "r2"] if c in fw_metrics.columns]
    print("=== Brain-AI Framework Backend Metrics ===")
    display(fw_metrics[display_cols].round(4))
else:
    print("No framework metrics produced — per-backend statuses:")
    for r in framework_output["results"]:
        status = r.metadata.get("status", "unknown")
        err = r.metadata.get("error", "")
        print(f"  {r.backend}: {status}  {'— ' + err[:150] if err else ''}")
    fw_metrics = None


,backend,task,status,prediction_payload_type,prediction_preview
0,autogluon_timeseries,forecasting,ok,dict,{'message': 'autogluon_timeseries adapter scaf...
1,statsforecast,forecasting,ok,dict,{'message': 'statsforecast adapter scaffolded'...
2,neuralforecast,forecasting,ok,dict,{'message': 'neuralforecast adapter scaffolded...
3,flaml,forecasting,ok,dict,"{'message': 'flaml adapter scaffolded', 'model..."


In [ ]:
# Compute metrics for direct benchmark models, then combine with framework metrics.
from brain_automl.model_zoo.time_series_ai.data_preparation import compute_forecast_metrics

direct_metric_rows = []
direct_model_cols = [c for c in predictions.columns if c not in ["date", "actual"]]

for col in direct_model_cols:
    valid_mask = predictions[col].notna()
    if valid_mask.sum() == 0:
        continue
    metrics = compute_forecast_metrics(
        y_train=train_df["y"].values,
        y_true=predictions.loc[valid_mask, "actual"].values,
        y_pred=predictions.loc[valid_mask, col].values,
    )
    direct_metric_rows.append({"backend": col, "source": "direct", **metrics})

direct_metrics_df = pd.DataFrame(direct_metric_rows)

# Merge framework + direct metrics into one comparison table.
if fw_metrics is not None and not fw_metrics.empty:
    fw_df = fw_metrics.copy()
    fw_df["source"] = "framework"
    combined_metrics = pd.concat([fw_df, direct_metrics_df], ignore_index=True, sort=False)
else:
    combined_metrics = direct_metrics_df

combined_metrics = combined_metrics.sort_values("rmse").reset_index(drop=True)
display_cols = [c for c in ["backend", "source", "mae", "rmse", "mape", "smape", "mase", "r2"] if c in combined_metrics.columns]

print("=== Combined Forecast Metrics — all backends (sorted by RMSE) ===")
display(combined_metrics[display_cols].round(4))


In [ ]:
# Direct forecasting benchmark models for last-1-year horizon.
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive, Naive

sf_train = train_df.copy()
sf_train["unique_id"] = selected_stock
sf_test = test_df.copy()
sf_test["unique_id"] = selected_stock

models = [
    AutoARIMA(season_length=5),
    SeasonalNaive(season_length=5),
    Naive(),
]

sf = StatsForecast(
    models=models,
    freq="B",
    n_jobs=1,
    fallback_model=Naive(),
)

sf_forecast = sf.forecast(df=sf_train[["unique_id", "ds", "y"]], h=len(sf_test))
sf_forecast = sf_forecast.rename(columns={"ds": "date"})

predictions = pd.DataFrame({
    "date": sf_test["ds"].values,
    "actual": sf_test["y"].values,
})

for col in sf_forecast.columns:
    if col != "unique_id" and col != "date":
        predictions[col] = sf_forecast[col].values

display(predictions.head())
print("Generated benchmark predictions for:", [c for c in predictions.columns if c not in ["date", "actual"]])

,date,actual,AutoARIMA,SeasonalNaive,Naive
0,2023-12-21,820.892822,806.780864,803.104492,806.438232
1,2023-12-22,813.178894,806.645893,806.219238,806.438232
2,2023-12-26,818.824402,806.416497,805.805542,806.438232
3,2023-12-27,828.971802,806.214976,804.442871,806.438232
4,2023-12-28,829.920837,806.262380,806.438232,806.438232


Generated benchmark predictions for: ['AutoARIMA', 'SeasonalNaive', 'Naive']


In [ ]:
# Decomposition-based model: STL + ARIMA for improved signal handling.
from statsmodels.tsa.forecasting.stl import STLForecast
from statsmodels.tsa.arima.model import ARIMA

stl_model = STLForecast(
    train_df.set_index("ds")["y"],
    ARIMA,
    period=5,
    model_kwargs={"order": (1, 1, 1)},
)
stl_res = stl_model.fit()
stl_pred = stl_res.forecast(len(test_df))

predictions["STL_ARIMA"] = stl_pred.values
display(predictions.tail())

/Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/t

,date,actual,AutoARIMA,SeasonalNaive,Naive,STL_ARIMA
247,2024-12-24,887.059631,857.083131,805.805542,806.438232,813.234580
248,2024-12-26,883.433655,857.297306,804.442871,806.438232,808.834652
249,2024-12-27,887.133667,857.511482,806.438232,806.438232,807.498862
250,2024-12-30,877.094360,857.725658,803.104492,806.438232,808.337762
251,2024-12-31,874.602966,857.939834,806.219238,806.438232,811.565356


In [ ]:
# Forecast vs actual visualization — framework backends + direct benchmarks on one chart.
COLORS = ["royalblue", "darkorange", "forestgreen", "crimson", "purple", "saddlebrown", "teal"]

fw_preds = framework_output["predictions"]  # cols: ds, actual, <backend_name>…
fw_backend_cols = [c for c in fw_preds.columns if c not in ["ds", "actual"]]

fig = go.Figure()

# Actual prices.
fig.add_trace(go.Scatter(
    x=fw_preds["ds"], y=fw_preds["actual"],
    name="Actual", line=dict(color="black", width=2),
))

# Framework backend forecasts (solid lines).
for i, col in enumerate(fw_backend_cols):
    fig.add_trace(go.Scatter(
        x=fw_preds["ds"], y=fw_preds[col],
        name=f"{col} (framework)",
        line=dict(color=COLORS[i % len(COLORS)], dash="dash"),
    ))

# Direct benchmark forecasts (dotted lines).
direct_cols = [c for c in predictions.columns if c not in ["date", "actual"]]
for j, col in enumerate(direct_cols):
    color_idx = (j + len(fw_backend_cols)) % len(COLORS)
    fig.add_trace(go.Scatter(
        x=predictions["date"], y=predictions[col],
        name=f"{col} (direct)",
        line=dict(color=COLORS[color_idx], dash="dot"),
    ))

fig.update_layout(
    title=f"Forecast vs Actual — {selected_stock}  (last {TEST_HORIZON} trading days)",
    xaxis_title="Date",
    yaxis_title="Close Price",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=560,
    hovermode="x unified",
    template="plotly_white",
)
fig.show()
